In [0]:
dbutils.widgets.dropdown(
    "select_store_db",
    "All",
    ["All", "DB1", "DB2", "DB3","DB4","DB5","DB6"]
)

dbutils.widgets.dropdown(
    "select_businessDate",
    "All",
    ["All", "2025-09-05", "2025-09-06", "2025-09-07", "2025-09-08"],
    "Select Date"
)

In [0]:
storage_files_map = {
   
    "DB1" : {
        "2025-09-05": "/Volumes/workspace/default/foodquest/DB1/ALBAIK - SQ DB01 - DUBAI MALL - 100500109-06-2025-1.json",
        "2025-09-06": "/Volumes/workspace/default/foodquest/DB1/ALBAIK - SQ DB01 - DUBAI MALL - 100500109-07-2025.json",
        "2025-09-07": "/Volumes/workspace/default/foodquest/DB1/ALBAIK - SQ DB01 - DUBAI MALL - 100500109-08-2025.json",
        "2025-09-08":""
    },

    "DB2" : {
        "2025-09-08":"",
        "2025-09-05": "/Volumes/workspace/default/foodquest/DB2/ALBAIK - SQ DB02 - DUBAI EXPO - 100500209-06-2025.json",
        "2025-09-06": "/Volumes/workspace/default/foodquest/DB2/ALBAIK - SQ DB02 - DUBAI EXPO - 100500209-07-2025.json",
        "2025-09-07": "/Volumes/workspace/default/foodquest/DB2/ALBAIK - SQ DB02 - DUBAI EXPO - 100500209-08-2025.json"
    },

    "DB3" : {
        "2025-09-05": "",
        "2025-09-06": "",
        "2025-09-07": "",
        "2025-09-08": ""
    },

    "DB4" : {
        "2025-09-06": "/Volumes/workspace/default/foodquest/DB4/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-06-2025.json",
        "2025-09-07": "/Volumes/workspace/default/foodquest/DB4/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-07-2025.json",
        "2025-09-08": "/Volumes/workspace/default/foodquest/DB4/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-08-2025.json"
    },

    "DB5" : {
        "2025-09-05": "/Volumes/workspace/default/foodquest/DB5/ALBAIK - SQ DB05 - AL MAJAZ - 100500509-06-2025.json",
        "2025-09-06": "/Volumes/workspace/default/foodquest/DB5/ALBAIK - SQ DB05 - AL MAJAZ - 100500509-07-2025.json",
        "2025-09-07": "/Volumes/workspace/default/foodquest/DB5/ALBAIK - SQ DB05 - AL MAJAZ - 100500509-08-2025.json",
        "2025-09-08":""
    },

    "DB6" : {
        "2025-09-05": "/Volumes/workspace/default/foodquest/DB6/ALBAIK - SQ DB06 - SHARJAH CITY CENTRE - 100500609-06-2025.json",
        "2025-09-06": "/Volumes/workspace/default/foodquest/DB6/ALBAIK - SQ DB06 - SHARJAH CITY CENTRE - 100500609-07-2025.json",
        "2025-09-07": "/Volumes/workspace/default/foodquest/DB6/ALBAIK - SQ DB06 - SHARJAH CITY CENTRE - 100500609-08-2025.json",
        "2025-09-08":""
    }
}

In [0]:
selected_store_db=dbutils.widgets.get("select_store_db")
selected_businessDate=dbutils.widgets.get("select_businessDate")

file_paths=[]

if selected_store_db=="All" and selected_businessDate=="All":
    for db, dates in storage_files_map.items():
        for date, path in dates.items():
          if path:
            file_paths.append(path)

elif selected_businessDate=="All":
    for date, path in storage_files_map[selected_store_db].items():
        if path:
            file_paths.append(path)

elif selected_store_db=="All":
    for db, dates in storage_files_map.items():
        path = dates.get("selected_businessDate","")
        if path:
            file_paths.append(path)

else:
    path=storage_files_map[selected_store_db].get(selected_businessDate, "")
    if path:
        file_paths.append(path)

if file_paths:
 df=spark.read.json(file_paths,multiLine=True)
 df.display()

else:
    print("No files found for the selected option")

print(file_paths)


In [0]:
# Item-wise-sales
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_valid=df.filter(
    col("isVoid")==False | col("isVoid").isNull())

#Explode kots (kitchen order tickets)

exploded_kots=df_valid.select(
    "billNumber",
    "businessDate",
    "deployment_name",
    "tab",
    explode(col("_kots")).alias("kot")
)

exploded_items=exploded_kots.select(
    "billNumber",
    "businessDate",
    "deployment_name",
    "tab",
    col("kot.created").alias("CreatedTime"),
    explode(col("kot.items")).alias("item")
)
exploded_kots.display()


exploded_items=exploded_items.withColumn(
    "TaxAmount",
    coalesce(
        aggregate(col("item.taxes"),
                  lit(0.0),
                  lambda acc,x: acc+x["tax_amount"]
        ),
        lit(0.0))
)
exploded_items.display()

taxes_df=exploded_items.select(
    explode("item.taxes").alias("tax")
)
taxes_df.display()



In [0]:
#map items

itemwise_sales=exploded_items.withColumn("tax", explode(col("item.taxes"))).select(
    col("billNumber").alias("BillNumber"),
    col("businessDate").alias("BuisnessDate"),
    col("item.uid.name").alias("ItemCode"),
    col("item.name").alias("ItemName"),
    col("item.quantity").alias("Quantity"),
    coalesce(col("item.rate"),col("item.originalRate")).alias("Rate"),
    col("item.subtotal").alias("Subtotal"),
    col("TaxAmount"),
    col("item.subTotalWithTax").alias("GrossAmount"),
    col("item.category.categoryName").alias("Category"),
    col("item.category.superCategory.superCategoryName").alias("SuperCategory"),
    col("item.category.categoryStation.stationName").alias("CategoryStation"),
    regexp_extract(col("deployment_name"), r"^([^-]+)", 1).alias("Brand"),
    regexp_extract(col("deployment_name"), r"(\d+)" ,1).alias("StoreCode"),
    col("deployment_name").alias("DeploymentName"),
    col("tab").alias("DeliveryType"),
    col("CreatedTime"),
    col("tax.name").alias("tax")
)

itemwise_sales.display()

# write as parquet file

itemwise_sales.repartition(1).write.mode("overwrite").parquet("/Volumes/workspace/default/foodquest/Item_wise_sales/")